In [ ]:
#Installing required packages
!pip install -q --upgrade transformers datasets accelerate bitsandbytes peft trl sentencepiece 2>/dev/null

In [ ]:
#Configuration
MODEL_ID = "unsloth/Llama-3.2-1B"
DATASET_PATH = "/kaggle/input/llama-fine-tuning-dataset/Fine_Tuning_Datasets.json"
OUTPUT_DIR = "/kaggle/working/llama_logic_classifier"

MAX_SEQ_LENGTH = 256
BATCH_SIZE = 2
GRAD_ACCUM_STEPS = 8
NUM_EPOCHS = 15
LEARNING_RATE = 2e-4
LORA_R = 16
LORA_ALPHA = 32

print("Configuration loaded!")

In [ ]:
#Importing libraries
import os
import json
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
#Loading dataset
print("Loading dataset...")
dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
print(f"Total examples: {len(dataset):,}")

def format_prompt(example):
    instruction = example.get("instruction", "Classify the logic of the sentence.")
    input_text = example.get("input", "")
    output = example.get("output", "")
    return {"text": f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n{output}"}

dataset = dataset.map(format_prompt)
print("Dataset formatted!")

In [ ]:
#Loading model for 4-bit quantization
print("Loading model with 4-bit quantization...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

# Loading tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map={"":0},
    trust_remote_code=True
)

model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)
print("Model loaded!")

In [ ]:
# Applying LoRA adapters
print("Applying LoRA adapters...")

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} ({100*trainable/total:.2f}%)")
print("LoRA applied!")

In [ ]:
print("Tokenizing...")

def tokenize_fn(examples):
    result = tokenizer(examples["text"], truncation=True, max_length=MAX_SEQ_LENGTH, padding="max_length")
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_dataset = dataset.map(tokenize_fn, batched=True, remove_columns=dataset.column_names)
tokenized_dataset.set_format("torch")
print(f"Tokenized {len(tokenized_dataset)} examples")

In [ ]:
print("="*60)
print("STARTING MANUAL TRAINING LOOP: ")
print("="*60)

from torch.utils.data import DataLoader
from transformers import get_scheduler, DataCollatorForLanguageModeling
from tqdm.auto import tqdm

# Create data collator
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_dataloader = DataLoader(tokenized_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=data_collator)

optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE)

num_update_steps_per_epoch = max(1, len(train_dataloader) // GRAD_ACCUM_STEPS)
num_training_steps = num_update_steps_per_epoch * NUM_EPOCHS
lr_scheduler = get_scheduler(
    name="cosine",
    optimizer=optimizer,
    num_warmup_steps=int(0.1 * num_training_steps),
    num_training_steps=num_training_steps,
)

scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

print(f"Total steps: {num_training_steps} | Steps per epoch: {num_update_steps_per_epoch}")

global_step = 0
model.train()

for epoch in range(NUM_EPOCHS):
    epoch_loss = 0.0
    progress_bar = tqdm(train_dataloader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}", leave=True)
    
    for step, batch in enumerate(progress_bar):
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
            outputs = model(**batch)
            loss = outputs.loss
            loss = loss / GRAD_ACCUM_STEPS
        scaler.scale(loss).backward()
        epoch_loss += loss.item()

        if (step + 1) % GRAD_ACCUM_STEPS == 0 or (step + 1) == len(train_dataloader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            lr_scheduler.step()
            global_step += 1
            
            # Update progress bar with current loss
            avg_loss = epoch_loss / (step + 1)
            progress_bar.set_postfix({"loss": f"{avg_loss:.4f}", "step": global_step})

    print(f"Epoch {epoch+1} completed | Avg Loss: {epoch_loss/len(train_dataloader):.4f}")

print("\n" + "="*60)
print("MANUAL TRAINING COMPLETE")
print("="*60)

In [ ]:
print("Saving model...")
adapter_path = os.path.join(OUTPUT_DIR, "lora_adapter")
model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)
print(f"Saved to: {adapter_path}")

In [ ]:
#Testing the fine-tuned model
print("Testing...")
test_sentences = [
    "If you're a photographer, keep all the necessary lens in the same area",
    "You must clean your brushes after every use",
    "First, gather your materials. Then, set up your workspace.",
]

model.eval()
for sentence in test_sentences:
    prompt = f"### Instruction:\nClassify the logic of the sentence into: Simple, Mandatory, Sequential, Conditional, Exclusive, Goal-based.\n\n### Input:\n{sentence}\n\n### Response:\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=10, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    pred = result.split("### Response:")[-1].strip().split("\n")[0]
    print(f"\n {sentence[:50]}...")
    print(f"{pred}")

print("\n Testing complete!")

In [ ]:
#Zipping the adapter for download
import shutil
zip_path = "/kaggle/working/lora_adapter.zip"
shutil.make_archive("/kaggle/working/lora_adapter", 'zip', adapter_path)
print(f"Model zipped: {zip_path}")
print(f"   Size: {os.path.getsize(zip_path) / 1e6:.1f} MB")
print("\nDownload from Output section!")